# Solution 01 — first producer

Reference answer for [`exercise_01_produce_single.ipynb`](../03_exercise/exercise_01_produce_single.ipynb). Read the exercise first; the comments here focus on *why* each line looks the way it does.

In [ ]:
from confluent_kafka import Producer
import json, time

BROKER = 'redpanda:29092'

# A single dict configures the producer. Many more knobs exist; we
# only override what we care about.
producer = Producer({
    'bootstrap.servers': BROKER,
    'client.id':         'python-producer',
})
print(f'Producer connected to {BROKER}')

### Check the topics

Both topics with 3 partitions, so that different keys can land on different partitions.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic, NewPartitions

PARTITIONS = 3
admin    = AdminClient({'bootstrap.servers': BROKER})
existing = admin.list_topics(timeout=5).topics

for name in ['rider-locations', 'orders']:
    if name not in existing:
        admin.create_topics([NewTopic(name, num_partitions=PARTITIONS)])[name].result()
        print(f'OK — created {name!r} with {PARTITIONS} partitions')
    elif len(existing[name].partitions) < PARTITIONS:
        admin.create_partitions([NewPartitions(name, PARTITIONS)])[name].result()
        print(f'OK — {name!r} grown to {PARTITIONS} partitions')
    else:
        print(f'OK — {name!r} already has {len(existing[name].partitions)} partitions')

## Step 2 — build the event

In [ ]:
topic_name    = 'rider-locations'
message_key   = 'rider_7'
message_value = json.dumps({
    'rider_id':  'rider_7',
    'lat':       47.0502,     # Lucerne main station
    'lon':       8.3093,
    'speed_kmh': 18.4,
    'timestamp': time.time(), # UNIX seconds
})
print(message_value)

## Step 3 — send + flush

Note `.encode('utf-8')` on key and value — librdkafka takes raw bytes. If you forget, you'll get a `TypeError`.

In [ ]:
def delivery_report(err, msg):
    if err:
        print(f'Delivery failed: {err}')
    else:
        print(f'Delivered to {msg.topic()} [{msg.partition()}] '
              f'offset {msg.offset()}')

producer.produce(
    topic_name,
    key=message_key.encode('utf-8'),
    value=message_value.encode('utf-8'),
    callback=delivery_report,
)
# flush() blocks until ALL queued messages have been ack'ed by the
# broker. Without it, the program could end before delivery happened.
producer.flush()

## Task A — second key on the same topic

Notice the partition number — it depends on the hash of the key. `rider_7` and `rider_3` land in different partitions, while `rider_5` would share a partition with `rider_7`: with only 3 partitions, hash collisions are common. Each key still always lands on the same partition.

In [ ]:
value_3 = json.dumps({
    'rider_id': 'rider_3', 'lat': 47.0469, 'lon': 8.3093,
    'speed_kmh': 22.0, 'timestamp': time.time(),
})
producer.produce('rider-locations', key=b'rider_3', value=value_3.encode(),
                 callback=delivery_report)
producer.flush()

## Task B — different topic

The order gets its own key, the order id: all later status changes of this order (`accepted`, `picked_up`, `delivered`) will use the same key and therefore stay in order.

In [ ]:
value_o = json.dumps({
    'order_id': 'order-4711', 'customer': 'Anna',
    'restaurant': 'Pizzeria Roma', 'total_chf': 32.50,
    'status': 'placed', 'timestamp': time.time(),
})
producer.produce('orders', key=b'order-4711', value=value_o.encode(),
                 callback=delivery_report)
producer.flush()

## Task C — broken broker

The error message is typically `KafkaError{code=_MSG_TIMED_OUT, …}` after our 4-second timeout. In production you would log this, count the failure rate (alerting if it spikes), and possibly buffer the message to a local dead-letter file or retry topic for human inspection.

In [ ]:
errors = []
def error_cb(err, msg):
    if err:
        errors.append(str(err))

p_broken = Producer({
    'bootstrap.servers':  'localhost:9999',
    'message.timeout.ms': 4000,
    'socket.timeout.ms':  2000,
})
p_broken.produce('rider-locations', key=b'rider_7', value=b'{"test": true}',
                 callback=error_cb)
p_broken.flush(timeout=5)
print(errors[0] if errors else 'unexpectedly delivered')